# Lab 5 · Model A versus Model B

**What you'll build:** `upscale` — the function the entire hypothesis test turns
on — and the diagnostic that decides whether the test could have worked at all.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labgrader import panel, grade_lab, TARGET, CLIMATE

df = panel()
print(df.shape, "rows x columns")
print("target column:", TARGET)

## 1. The question, and the trap in answering it

The hypothesis: *scale-free downscaled climate (~375–750 m) predicts corridor
water stress better than coarse grid climate does.*

The obvious experiment is to grab a coarse climate product and race it against
ClimateBC. **Don't.** Two different products differ in their source stations,
their interpolation, their reference period, their bias correction — and their
resolution. If one wins you cannot say which of those five things won.

So instead: take ClimateBC and **blur it yourself**. Average the values within
coarse cells and hand each corridor its cell's mean.

- **Model A** — ClimateBC at each corridor's own point and elevation.
- **Model B** — *the same numbers*, averaged into 4 km cells.

Same data, same target, same folds, same algorithm, same seed. Resolution is the
only thing that moves. A narrower question answered cleanly beats a broader one
answered uninterpretably.

### ✏️ Assignment 1 — `upscale`

Replace each row's features with the mean over its **(cell, year)** group.

1. Cell id from the projected coordinates:
   `np.floor(x_m / cell_m).astype(int).astype(str) + "_" + same for y_m`.
2. Group by `[cell, "year"]` and `.transform("mean")` the feature columns.
3. Return a copy of `df` with those columns replaced.

Why group by cell **and year**, not cell alone? Because averaging across years
too would flatten the temporal signal as well — and then Model B would differ
from Model A in *two* ways, which is exactly the confound this design exists to
avoid. Only the spatial detail may be destroyed.

In [ ]:
def upscale(df, features, cell_m=4000.0):
    """Give every row its coarse cell's mean, computed within its own year."""
    # >>> YOUR TURN
    raise NotImplementedError

In [ ]:
FEATS = ["PPT_sm", "Tmax_sm", "CMD_sm"]
coarse = upscale(df, FEATS, 4000.0)

one = df["year"] == 2023
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, src, name in zip(axes, (df, coarse), ("Model A — scale-free", "Model B — 4 km cells")):
    s = ax.scatter(src.loc[one, "x_m"], src.loc[one, "y_m"],
                   c=src.loc[one, "Tmax_sm"], cmap="inferno", s=22)
    ax.set_aspect("equal"); ax.set_title(name); fig.colorbar(s, ax=ax, shrink=.8)
fig.suptitle("Summer Tmax over Surrey's corridors, 2023")
plt.tight_layout(); plt.show()

In [ ]:
# Sanity: the year means must be untouched. Only space was blurred.
pd.DataFrame({
    "fine":   df.groupby("year")[FEATS].mean().stack(),
    "coarse": coarse.groupby("year")[FEATS].mean().stack(),
}).round(6)

## 2. Did the blur actually do anything?

This is the step that saves the project from a wrong conclusion.

If the upscale barely changed the predictors, then A and B are nearly the *same
model*, and finding no difference between them is **arithmetic, not evidence**.
You have to check before you interpret.

Measure it: how much of each feature's **spatial** variance did the blur destroy?

1. Remove the year effect: `within = df.groupby("year")[f].transform("mean")`.
2. Spatial variance of the fine data: `mean((fine - within)**2)`.
3. Same for the coarse data.
4. Fraction removed: `1 - coarse_var / fine_var`.

### ✏️ Assignment 2 — `var_removed`

In [ ]:
def var_removed(fine, coarse, feature):
    """Fraction of `feature`'s spatial variance destroyed by the upscale."""
    # >>> YOUR TURN
    raise NotImplementedError

In [ ]:
allc = upscale(df, CLIMATE, 4000.0)
tab = pd.DataFrame([{"feature": f, "var_removed": var_removed(df, allc, f)}
                    for f in CLIMATE]).sort_values("var_removed", ascending=False)
print(tab.round(3).to_string(index=False))
print(f"\nmedian: {tab['var_removed'].median()*100:.0f}%")
print("gate for a meaningful contrast: 30%")

> **The blur gate fails over Surrey.** The median comes in around 12%, well under
> the 30% the design requires. Most of ClimateBC's variation across 30 flat
> kilometres sits *between* 4 km cells and survives the averaging untouched.
>
> So Model A and Model B are nearly the same model. Whatever the comparison
> returns in Lab 6, it cannot be read as evidence about resolution.
>
> This is precondition **2** failing. Lab 1 already showed precondition 1 failing
> — the predictors and the target vary along different axes, so nothing has
> skill. Two independent reasons, and neither is fixable by adding corridors.
>
> For contrast: the Fraser Valley transect at 25 km removes **48%**, clearing the
> gate — which is exactly why Phase 3b exists and why its verdict is FALSIFIED
> rather than INCONCLUSIVE.

In [ ]:
# Cell size is a dial. Watch the contrast climb — and notice how coarse you have
# to go before Surrey clears the gate at all.
for cm in (1000, 2000, 4000, 8000, 16000, 25000):
    c = upscale(df, CLIMATE, float(cm))
    med = np.median([var_removed(df, c, f) for f in CLIMATE])
    flag = "clears 30%" if med >= .30 else ""
    print(f"  {cm:>6,} m  ->  {med*100:5.1f}%   {flag}")

---
## Grade it

In [ ]:
grade_lab(5, globals())

### What you should be able to say out loud

- Model B is Model A blurred, **not a different product** — that is how the
  design isolates resolution from everything else.
- Averaging within (cell, **year**) destroys space while leaving time intact.
- The blur gate is a **precondition**, checked before the result is interpreted.
- Surrey removes ~12%; the transect at 25 km removes 48%. Same code, different
  extent, different verdict.